# Qwen3-0.6B DoRA+ — 10k Fine-Tuning

This notebook runs the full-dataset experiment: **DoRA+**, **Qwen3-0.6B**, and configuration **D** (full attention + MLP, rank 32).

The runtime-only configuration below is based on config D, with batch size **16**, gradient accumulation **1**, effective batch **16**, learning rate **5e-5**, and **4 epochs**. It does not modify repository files.

## 1. Clone repository

In [1]:
import os

REPO_URL = "https://github.com/kon172verma/intent-classifier.git"
REPO_DIR = "/content/intent-classifier"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

print(f"Repo at: {REPO_DIR}")

Repo at: /content/intent-classifier


## 2. Install dependencies

In [2]:
%pip install -q \
    torch \
    "torchao>=0.16.0" \
    transformers \
    accelerate \
    "peft>=0.14.0" \
    trl \
    datasets \
    bitsandbytes \
    huggingface_hub \
    python-dotenv \
    sentencepiece \
    protobuf

print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 51.6 MB/s eta 0:00:00
Dependencies installed.


## 3. Hugging Face authentication

Store your `HF_TOKEN` in **Colab Secrets** (key icon in the left sidebar).
Required for gated models (`llama3.2-1b`) and pushing adapters.

In [3]:
import os

try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("WARNING: HF_TOKEN secret is empty.")
except Exception as e:
    print(f"Not running in Colab or secret missing: {e}")

HF_TOKEN loaded from Colab Secrets.


## 4. GPU environment check

In [4]:
import subprocess
import platform
import torch

print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

Python  : 3.13.15
PyTorch : 2.11.0+cu128
CUDA    : 12.8
Device  : NVIDIA L4
VRAM    : 23.7 GB


## 5. Set the 10k runtime configuration

This changes only the active Colab kernel's in-memory copy of config D. The repository's source files remain untouched.

In [5]:
import runpy
import sys
from pathlib import Path

SRC_DIR = Path(REPO_DIR) / "finetune_DoRAplus" / "src"
DATA_DIR = Path(REPO_DIR) / "finetune_DoRAplus" / "data"
MODEL = "qwen3-0.6b"
CONFIG = "D"
DATASET_SIZE = "10k"
DEVICE = "cuda"
PER_DEVICE_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from finetune_lib import LORA_CONFIGS, LORAPLUS_CONFIGS

runtime_config = {
    **LORA_CONFIGS[CONFIG],
    "per_device_train_batch_size": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
}
LORA_CONFIGS[CONFIG] = runtime_config
LORAPLUS_CONFIGS[CONFIG] = {**LORAPLUS_CONFIGS[CONFIG], **runtime_config}

print(f"Model: {MODEL}")
print(f"Config: {CONFIG} (same adapter scope as repository config D)")
print(f"Per-device batch: {PER_DEVICE_BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch: {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate: {runtime_config['learning_rate']:.1e}")
print(f"Epochs: {runtime_config['num_train_epochs']}")

# Leave both as None after running section 8: the newest local registry entry is used.
# To evaluate a previously published adapter in a fresh Colab session, set both
# values from its HF subfolder, for example version="v2.0" and
# timestamp="20260827-213047".
ADAPTER_VERSION: str | None = "v2.1"
ADAPTER_TIMESTAMP: str | None = "20260827-213047"

def hub_adapter_args() -> list[str]:
    if (ADAPTER_VERSION is None) != (ADAPTER_TIMESTAMP is None):
        raise ValueError("Set both ADAPTER_VERSION and ADAPTER_TIMESTAMP, or neither.")
    if ADAPTER_VERSION is None:
        return []
    return ["--version", ADAPTER_VERSION, "--timestamp", ADAPTER_TIMESTAMP]

def run_repo_entrypoint(script_name: str, arguments: list[str]) -> None:
    previous_argv = sys.argv
    try:
        sys.argv = [str(SRC_DIR / script_name), *arguments]
        runpy.run_path(str(SRC_DIR / script_name), run_name="__main__")
    finally:
        sys.argv = previous_argv

Model: qwen3-0.6b
Config: D (same adapter scope as repository config D)
Per-device batch: 16
Gradient accumulation: 1
Effective batch: 16
Learning rate: 5.0e-05
Epochs: 4


## 6. Prepare the 10k 80/10/10 data split

In [6]:
import subprocess

prepare_cmd = [
    sys.executable, "-u", str(SRC_DIR / "prepare_doraplus_data.py"),
    "--dataset-size", DATASET_SIZE,
    "--out-dir", str(DATA_DIR),
]
subprocess.run(prepare_cmd, cwd=REPO_DIR, check=True)

CompletedProcess(args=['/usr/bin/python3', '-u', '/content/intent-classifier/finetune_DoRAplus/src/prepare_doraplus_data.py', '--dataset-size', '10k', '--out-dir', '/content/intent-classifier/finetune_DoRAplus/data'], returncode=0)

## 7. Optional 10-step smoke test

Run this on a fresh Colab runtime before full training. It uses the same runtime-only configuration and does not upload an adapter.

In [ ]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    run_repo_entrypoint(
        "doraplus_train.py",
        [
            "--model", MODEL,
            "--lora-config", CONFIG,
            "--dataset-size", DATASET_SIZE,
            "--device", DEVICE,
            "--gradient-checkpointing",
            "--smoke-test",
            "--no-push",
        ],
    )


  DoRA+ Training — qwen3-0.6b_D_10k
  Model        : Qwen/Qwen3-0.6B
  LoRA config  : D — Heavy — full attention + MLP, rank 32
  Dataset      : 10k
  Device       : cuda
  Adapter dest : /content/intent-classifier/finetune_DoRAplus/adapters/qwen3-0.6b_D_10k
  HF repo      : kon172verma/intent-classifier-experiments/v2.0/qwen3-0.6b_DoRA+_D_10k_<timestamp>
  Mode         : SMOKE TEST (10 steps only)

  Train : 8000 examples
  Val   : 1000 examples

  Loading tokenizer: Qwen/Qwen3-0.6B
  Loading model:     Qwen/Qwen3-0.6B


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


  Trainable params : 20,529,152  (3.330%)
  Total params     : 616,579,072
  Grad checkpoint  : enabled

  Tokenizing datasets...

  Effective batch  : 16
  Steps / epoch    : 500
  Total steps      : 10 (smoke)
  Eval every       : 10 steps
  LoRA+ optimizer  : ratio=8  lr_A=5.0e-05  lr_B=4.0e-04


Truncating train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

...

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
No log,2.352274,0,0.338231,0.000000,0.763667


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


  Step 0 — train_loss=2.1566  val_loss=2.352273941040039  train_acc=0.3  val_acc=0.34

  Starting training (smoke-test: 10 steps)...



Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,1.031419,0.135941,0.144142,55094.000000,0.972000



  [Accuracy] step=10  train=0.8600  val=0.9300

  Training complete in 200.2s  |  Peak VRAM: 4583 MB

  TRAINING COMPLETE — qwen3-0.6b_D_10k
  Final train loss : 1.0314
  Final val loss   : 0.1359
  Final val acc    : 0.9300
  Final train acc  : 0.8600
  Training time    : 200.2s
  Peak VRAM        : 4583 MB
  Training report  : /content/intent-classifier/finetune_DoRAplus/reports_training/qwen3-0.6b_D_10k_20260827_184332.json



## 8. Train the adapter

This runs the full 10k experiment and uploads the adapter and report to Hugging Face.

In [ ]:
run_repo_entrypoint(
    "doraplus_train.py",
    [
        "--model", MODEL,
        "--lora-config", CONFIG,
        "--dataset-size", DATASET_SIZE,
        "--device", DEVICE,
        "--gradient-checkpointing",
    ],
)


  DoRA+ Training — qwen3-0.6b_D_10k
  Model        : Qwen/Qwen3-0.6B
  LoRA config  : D — Heavy — full attention + MLP, rank 32
  Dataset      : 10k
  Device       : cuda
  Adapter dest : /content/intent-classifier/finetune_DoRAplus/adapters/qwen3-0.6b_D_10k
  HF repo      : kon172verma/intent-classifier-experiments/v2.0/qwen3-0.6b_DoRA+_D_10k_<timestamp>

  Train : 8000 examples
  Val   : 1000 examples

  Loading tokenizer: Qwen/Qwen3-0.6B
  Loading model:     Qwen/Qwen3-0.6B


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


  Trainable params : 20,529,152  (3.330%)
  Total params     : 616,579,072
  Grad checkpoint  : enabled

  Tokenizing datasets...

  Effective batch  : 16
  Steps / epoch    : 500
  Total steps      : 2000
  Eval every       : 250 steps
  LoRA+ optimizer  : ratio=8  lr_A=5.0e-05  lr_B=4.0e-04


Truncating train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

...

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
No log,2.352274,0,0.338231,0.000000,0.763667


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


  Step 0 — train_loss=2.1566  val_loss=2.352273941040039  train_acc=0.3  val_acc=0.34

  Starting training...



Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
250,0.129668,0.042427,0.027871,1516296.000000,0.994333
500,0.014683,0.012299,0.011986,3035617.000000,0.996667
750,0.016262,0.001208,0.003733,4550488.000000,0.999667
1000,0.003705,0.004278,0.000578,6071234.000000,0.999667
1250,0.000932,0.002246,0.000366,7586814.000000,0.999667



  [Accuracy] step=250  train=1.0000  val=0.9700

  [Accuracy] step=500  train=1.0000  val=0.9900

  [Accuracy] step=750  train=1.0000  val=1.0000

  [Accuracy] step=1000  train=1.0000  val=1.0000

  [Accuracy] step=1250  train=1.0000  val=1.0000

  Training complete in 9873.2s  |  Peak VRAM: 4593 MB

  Saving adapter locally → /content/intent-classifier/finetune_DoRAplus/adapters/qwen3-0.6b_D_10k
  Pushing adapter to HF → kon172verma/intent-classifier-experiments/v2.0/qwen3-0.6b_DoRA+_D_10k_20260827-213047
  Adapter pushed successfully.
  Report pushed  : kon172verma/intent-classifier-experiments/reports/v2.0/doraplus/reports_training/qwen3-0.6b_D_10k_20260827_213058.json

  TRAINING COMPLETE — qwen3-0.6b_D_10k
  Final train loss : 0.0330
  Final val loss   : 0.0022
  Final val acc    : 1.0000
  Final train acc  : 1.0000
  Training time    : 9873.2s
  Peak VRAM        : 4593 MB
  Training report  : /content/intent-classifier/finetune_DoRAplus/reports_training/qwen3-0.6b_D_10k_20260827

## 9. Validate the published adapter

The training cell saves the final adapter locally and publishes it to the Hugging Face experiments repository. This evaluation loads that published adapter and uploads its validation report to the same repository.

In [7]:
run_repo_entrypoint(
    "doraplus_validate.py",
    [
        "--model", MODEL,
        "--lora-config", CONFIG,
        "--dataset-size", DATASET_SIZE,
        "--split", "val",
        "--device", DEVICE,
        *hub_adapter_args(),
    ],
)


  DoRA+ Evaluation — qwen3-0.6b_D_10k
  Split    : val  (1000 examples)
  Source   : kon172verma/intent-classifier-experiments/v2.1/qwen3-0.6b_DoRA+_D_10k_20260827-213047
  Device   : cuda

  Loading base model: Qwen/Qwen3-0.6B


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

  Loading adapter from HF: kon172verma/intent-classifier-experiments/v2.1/qwen3-0.6b_DoRA+_D_10k_20260827-213047


adapter_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

v2.1/qwen3-0.6b_DoRA+_D_10k_20260827-213(…): reconstructing file:   0%|          |  0.00B / 82.2MB            

v2.1/qwen3-0.6b_DoRA+_D_10k_20260827-213(…): downloading bytes:           |  0.00B            

  [  20/1000]  running_acc=1.000
  [  40/1000]  running_acc=1.000
  [  60/1000]  running_acc=1.000
  [  80/1000]  running_acc=1.000
  [ 100/1000]  running_acc=1.000
  [ 120/1000]  running_acc=1.000
  [ 140/1000]  running_acc=1.000
  [ 160/1000]  running_acc=1.000
  [ 180/1000]  running_acc=1.000
  [ 200/1000]  running_acc=1.000
  [ 220/1000]  running_acc=1.000
  [ 240/1000]  running_acc=1.000
  [ 260/1000]  running_acc=1.000
  [ 280/1000]  running_acc=1.000
  [ 300/1000]  running_acc=1.000
  [ 320/1000]  running_acc=1.000
  [ 340/1000]  running_acc=1.000
  [ 360/1000]  running_acc=1.000
  [ 380/1000]  running_acc=1.000
  [ 400/1000]  running_acc=1.000
  [ 420/1000]  running_acc=1.000
  [ 440/1000]  running_acc=1.000
  [ 460/1000]  running_acc=1.000
  [ 480/1000]  running_acc=1.000
  [ 500/1000]  running_acc=1.000
  [ 520/1000]  running_acc=1.000
  [ 540/1000]  running_acc=1.000
  [ 560/1000]  running_acc=0.998
  [ 580/1000]  running_acc=0.998
  [ 600/1000]  running_acc=0.998
  [ 620/10

## 10. Optional locked test evaluations

Set the switch to `True` only after reviewing validation results. It evaluates both the full 1,000-example test split and the 100-example `sample_0001` anchor split used by the earlier experiments. Both reports are uploaded to the Hugging Face experiments repository.

In [8]:
RUN_TEST_EVALUATION = True

if RUN_TEST_EVALUATION:
    for split in ("test", "test_anchor"):
        run_repo_entrypoint(
            "doraplus_validate.py",
            [
                "--model", MODEL,
                "--lora-config", CONFIG,
                "--dataset-size", DATASET_SIZE,
                "--split", split,
                "--device", DEVICE,
                *hub_adapter_args(),
            ],
        )


  DoRA+ Evaluation — qwen3-0.6b_D_10k
  Split    : test  (1000 examples)
  Source   : kon172verma/intent-classifier-experiments/v2.1/qwen3-0.6b_DoRA+_D_10k_20260827-213047
  Device   : cuda

  Loading base model: Qwen/Qwen3-0.6B


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  Loading adapter from HF: kon172verma/intent-classifier-experiments/v2.1/qwen3-0.6b_DoRA+_D_10k_20260827-213047
  [  20/1000]  running_acc=1.000
  [  40/1000]  running_acc=1.000
  [  60/1000]  running_acc=1.000
  [  80/1000]  running_acc=1.000
  [ 100/1000]  running_acc=1.000
  [ 120/1000]  running_acc=1.000
  [ 140/1000]  running_acc=1.000
  [ 160/1000]  running_acc=1.000
  [ 180/1000]  running_acc=1.000
  [ 200/1000]  running_acc=1.000
  [ 220/1000]  running_acc=1.000
  [ 240/1000]  running_acc=1.000
  [ 260/1000]  running_acc=1.000
  [ 280/1000]  running_acc=1.000
  [ 300/1000]  running_acc=1.000
  [ 320/1000]  running_acc=1.000
  [ 340/1000]  running_acc=1.000
  [ 360/1000]  running_acc=1.000
  [ 380/1000]  running_acc=1.000
  [ 400/1000]  running_acc=1.000
  [ 420/1000]  running_acc=1.000
  [ 440/1000]  running_acc=1.000
  [ 460/1000]  running_acc=1.000
  [ 480/1000]  running_acc=1.000
  [ 500/1000]  running_acc=1.000
  [ 520/1000]  running_acc=1.000
  [ 540/1000]  running_acc=1.

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  Loading adapter from HF: kon172verma/intent-classifier-experiments/v2.1/qwen3-0.6b_DoRA+_D_10k_20260827-213047
  [  20/100]  running_acc=1.000
  [  40/100]  running_acc=1.000
  [  60/100]  running_acc=1.000
  [  80/100]  running_acc=1.000
  [ 100/100]  running_acc=1.000
  Report pushed  : kon172verma/intent-classifier-experiments/reports/v2.1/doraplus/reports_test/qwen3-0.6b_D_10k_test_anchor_20260828_104534.json

  Accuracy   : 1.0000  (100/100)
  Avg latency: 300.9 ms
  Peak memory: 1395 MB
  Report     : /content/intent-classifier/finetune_DoRAplus/reports_test/qwen3-0.6b_D_10k_test_anchor_20260828_104534.json


## 11. Download reports

Creates a ZIP containing every locally generated training, validation, and test report. The final adapter is already saved locally and uploaded by the training cell; reports are also uploaded when cells 9 and 10 run.

In [9]:
import shutil
from pathlib import Path

try:
    from google.colab import files  # type: ignore

    doraplus_dir = Path(REPO_DIR) / "finetune_DoRAplus"
    report_dirs = {
        "reports_training": doraplus_dir / "reports_training",
        "reports_validation": doraplus_dir / "reports_validation",
        "reports_test": doraplus_dir / "reports_test",
    }
    staging = Path("/content/_doraplus_reports_staging")
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir()

    for name, src in report_dirs.items():
        if src.exists() and any(src.glob("*.json")):
            shutil.copytree(src, staging / name)
        else:
            print(f"Skipping {name}: no JSON reports found")

    archive = "/content/doraplus_10k_reports"
    shutil.make_archive(archive, "zip", staging)
    print(f"Created {archive}.zip")
    files.download(f"{archive}.zip")
except ImportError:
    print(f"Not running in Colab. Reports are in {REPO_DIR}/finetune_DoRAplus/reports_*")

Created /content/doraplus_10k_reports.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>